# Paths to Causality — Demo trên Google Colab (T4 GPU)

Chạy toàn bộ demo tự host (Streamlit + Neo4j + Hetionet + Ollama) trên một máy Colab,
tận dụng **T4 GPU** để suy luận cross-encoder (RoBERTa) đã huấn luyện ở Module 2.

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`, rồi chạy các cell THEO THỨ TỰ từ trên xuống.
Mỗi cell đều an toàn khi chạy lại (idempotent).

## 1. Clone repo + cài dependencies

In [ ]:
%cd /content
!rm -rf Reproduce_Paths-to-Causality_KDD-25
!git clone -b demo https://github.com/KhangNLY/Reproduce_Paths-to-Causality_KDD-25.git
%cd /content/Reproduce_Paths-to-Causality_KDD-25/demo

In [ ]:
# Dependencies theo requirements.txt của demo (torch/transformers Colab có sẵn;
# 2 dòng comment dưới để dự phòng nếu runtime thiếu).
!pip -q install streamlit neo4j requests pandas numpy scikit-learn xgboost pyvis joblib python-dotenv 2>&1 | tail -1
# !pip -q install torch transformers

import importlib
for m in ("torch", "transformers", "streamlit", "neo4j", "xgboost", "pandas", "sklearn"):
    ok = importlib.util.find_spec(m) is not None
    print(f"  {m:12} {'OK' if ok else 'THIẾU'}")
import torch
print("CUDA:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "chỉ CPU")

## 2. Tải weights đã huấn luyện từ Google Drive
5 bộ: 4 checkpoint cross-encoder RoBERTa (ADE / COMAGC / GENEC / SemEval) + 1 bộ XGBoost + n-gram (all tasks). Tổng ~2.5 GB — mất vài phút.

In [ ]:
%cd /content/Reproduce_Paths-to-Causality_KDD-25
!mkdir -p saved_models

# 1. ptc_weights_cross_encoder_ade_roberta_best.zip
!gdown 1UOTaEeB8yv10WjsqYFtETYnNSgJpc-vo -O saved_models/ade_roberta.zip
# 2. ptc_weights_cross_encoder_comagc_roberta_best.zip
!gdown 1NDmPTK68DgiMZWCHYjkodQtqY_IGKjQp -O saved_models/comagc_roberta.zip
# 3. ptc_weights_cross_encoder_gene_roberta_best.zip
!gdown 1AksCE7gRf5IUKssJ1me6WmYNuSSxtF13 -O saved_models/gene_roberta.zip
# 4. ptc_weights_cross_encoder_semeval_roberta_best.zip
!gdown 1ul9J3RsbZri6TlPRgc0lztXhk0XXHYE1 -O saved_models/semeval_roberta.zip
# 5. ptc_weights_xgboost_all_tasks_best_20261104.zip
!gdown 1QHj-z_yAGC8DTEUgmT3hnDKNjmZFIdWg -O saved_models/xgboost_all_tasks.zip

## 3. Giải nén weights về đường dẫn cố định của demo
Demo load checkpoint từ đường dẫn cố định (xem README *Model checkpoints: fixed locations*):
cross-encoder tại `demo/models/ce/`, XGBoost artifact tại `demo/models/repo_ranker/`.

In [ ]:
%cd /content/Reproduce_Paths-to-Causality_KDD-25
import zipfile, os
saved_models_dir = "/content/Reproduce_Paths-to-Causality_KDD-25/saved_models"
for file in os.listdir(saved_models_dir):
    if file.endswith(".zip"):
        file_path = os.path.join(saved_models_dir, file)
        extract_path = os.path.join(saved_models_dir, file.replace(".zip", ""))
        print(f"Đang giải nén {file} vào {extract_path}...")
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
print("=== Giải nén hoàn tất!")
!ls -lh saved_models

In [ ]:
# Sắp weights vào ĐÚNG đường dẫn cố định của demo.
import shutil, json
from pathlib import Path

root = Path("/content/Reproduce_Paths-to-Causality_KDD-25")
ce_root = root / "demo" / "models" / "ce"
ce_root.mkdir(parents=True, exist_ok=True)

# (a) Cross-encoder: mọi thư mục chứa model.safetensors -> demo/models/ce/<tên gốc>
ce_files = list(Path(saved_models_dir).rglob("model.safetensors"))
print(f"Thấy {len(ce_files)} checkpoint cross-encoder")
for f in ce_files:
    src, dst = f.parent, ce_root / f.parent.name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    meta = json.loads((dst / "train_meta.json").read_text()) if (dst / "train_meta.json").exists() else {}
    print(f"  {f.parent.name}  (ranker={meta.get('ranker','?')}, task={meta.get('task','?')}, "
          f"NDCG@5={meta.get('test_corrected_top5_NDCG@5','?')})")

# (b) XGBoost + n-gram: demo chỉ nạp MỘT artifact -> chọn dir có corrected NDCG@5 cao nhất
xgb_dst = root / "demo" / "models" / "repo_ranker"
cfgs = [p.parent for p in Path(saved_models_dir).rglob("xgb_ranker.json")]
if cfgs:
    def score_of(d):
        for name in ("metrics.json", "train_meta.json"):
            f = d / name
            if f.exists():
                try:
                    return float(json.loads(f.read_text()).get("test_corrected_top5_NDCG@5", -1))
                except Exception:
                    pass
        return -1
    src = max(cfgs, key=score_of)
    xgb_dst.mkdir(parents=True, exist_ok=True)
    for name in ("xgb_ranker.json", "ngram_embeddings.npz", "config.json"):
        if (src / name).exists():
            shutil.copy2(src / name, xgb_dst / name)
    print(f"XGBoost artifact (NDCG tốt nhất trong zip): {src.name} -> {xgb_dst}")
else:
    print("Không thấy artifact XGBoost (demo vẫn chạy với CE/local XGB).")

!ls /content/Reproduce_Paths-to-Causality_KDD-25/demo/models/ce | head

## 4. Cài Neo4j standalone + nạp Hetionet

In [ ]:
import subprocess
# Cài Neo4j 5 community + start DETACHED + chờ sẵn sàng (rerun-safe).
import pathlib, subprocess, time

if not pathlib.Path("/opt/neo4j/bin/neo4j").exists():
    !wget -q https://neo4j.com/artifact.php?name=neo4j-community-5.26.0-unix.tar.gz -O /tmp/neo4j.tar.gz
    !tar -xzf /tmp/neo4j.tar.gz -C /opt
    !mv /opt/neo4j-community-5.26.0 /opt/neo4j
else:
    print("Neo4j đã cài, bỏ qua.")

# Binding + GIỚI HẠN BỘ NHỚ cho vừa RAM ~12GB của Colab (tránh OOM kill)
conf = pathlib.Path("/opt/neo4j/conf/neo4j.conf")
txt = conf.read_text().replace("#server.default_listen_address=0.0.0.0",
                               "server.default_listen_address=0.0.0.0")
lines = [l for l in txt.splitlines()
         if not l.startswith(("server.memory.heap.initial_size",
                              "server.memory.heap.max_size",
                              "server.memory.pagecache.size"))]
conf.write_text("\n".join(lines) +
    "\nserver.memory.heap.initial_size=512m"
    "\nserver.memory.heap.max_size=1G"
    "\nserver.memory.pagecache.size=1G\n")

# Password (chỉ tác dụng trước lần start đầu)
if not pathlib.Path("/opt/neo4j/data/dbms/auth.ini").exists():
    !/opt/neo4j/bin/neo4j-admin dbms set-initial-password local-demo-change-me

# Stop instance cũ (nếu còn) rồi start DETACHED bằng console mode:
# process start qua !shell sẽ bị Colab giết khi cell kết thúc.
subprocess.run(["/opt/neo4j/bin/neo4j", "stop"], capture_output=True); time.sleep(5)
_log = open("/tmp/neo4j-console.log", "w")
subprocess.Popen(["/opt/neo4j/bin/neo4j", "console"],
                 start_new_session=True, stdout=_log, stderr=subprocess.STDOUT, cwd="/opt/neo4j")

# Poll tới khi bolt nhận kết nối (lần đầu khởi tạo DB có thể mất 40-90s)
ready = False
for i in range(36):
    time.sleep(5)
    r = subprocess.run(["/opt/neo4j/bin/cypher-shell", "-u", "neo4j",
                        "-p", "local-demo-change-me", "RETURN 1 AS ok"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        ready = True; print(f"✓ Neo4j sẵn sàng sau ~{(i+1)*5}s"); break
    print(f"đang chờ Neo4j... ({(i+1)*5}s)")
if not ready:
    print("CHƯA xong — log cuối:"); print(open("/tmp/neo4j-console.log").read()[-1500:])

In [ ]:
# Tải Hetionet v1.0 + import vào Neo4j (~1-3 phút; chạy lại sẽ bỏ qua data đã tải)
%cd /content/Reproduce_Paths-to-Causality_KDD-25/demo
import os
os.environ["NEO4J_URI"] = "bolt://localhost:7687"   # script đọc env này

!bash scripts/download_hetionet.sh
!python scripts/load_hetionet.py
!/opt/neo4j/bin/cypher-shell -u neo4j -p local-demo-change-me "MATCH (n:Entity) RETURN count(n) AS nodes" 

## 5. Cài Ollama + pull `qwen2.5:7b`
Model ~4.7 GB — tải vài phút. Server start detached để sống qua các cell sau.

In [ ]:
import subprocess
# Image Colab thiếu zstd — script cài Ollama cần nó để giải nén.
import pathlib, subprocess, time
if not pathlib.Path("/usr/local/bin/ollama").exists():
    !apt-get install -y -q zstd
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("Ollama đã cài, bỏ qua.")

# start DETACHED (Colab giết process nền của cell đã kết thúc)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                 start_new_session=True)

# chờ server nhận kết nối rồi pull
import requests
for i in range(20):
    time.sleep(2)
    try:
        requests.get("http://localhost:11434/"); break
    except Exception:
        pass
!ollama pull qwen2.5:7b
!ollama list

## 6. Khởi động Streamlit + tunnel Cloudflare
Cell in ra **URL công khai** `https://xxx.trycloudflare.com` — mở URL đó để dùng demo.

In [ ]:
import subprocess, time, re, socket, pathlib, requests

%cd /content/Reproduce_Paths-to-Causality_KDD-25/demo

# .env cho môi trường Colab
env = """NEO4J_URI=bolt://localhost:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=local-demo-change-me
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=qwen2.5:7b
MAX_HOPS=4
MAX_PATHS=30
TOP_K_PATHS=3
RANKER_PATH=models/xgb_ranker.json
REPO_CE_DIR=models/ce
REPO_RANKER_DIR=models/repo_ranker
"""
open(".env", "w").write(env)

# cloudflared (tải 1 lần)
if not pathlib.Path("/usr/local/bin/cloudflared").exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Dọn process cũ chiếm cổng (rerun-safe)
subprocess.run("pkill -f streamlit; pkill -f cloudflared; sleep 2", shell=True)
time.sleep(3)

# Streamlet DETACHED + chờ THẬT SỰ lắng nghe cổng
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"],
                 cwd="/content/Reproduce_Paths-to-Causality_KDD-25/demo",
                 start_new_session=True,
                 stdout=open("/tmp/streamlit.log", "w"), stderr=subprocess.STDOUT)
up = False
for i in range(30):
    time.sleep(2)
    try:
        socket.create_connection(("localhost", 8501), timeout=2).close()
        up = True; print(f"✓ Streamlit lắng nghe 8501 sau ~{(i+1)*2}s"); break
    except OSError:
        pass
if not up:
    print("Streamlit chưa lên — log:"); print(open("/tmp/streamlit.log").read()[-1500:])

# Tunnel: LỆNH ĐÚNG là "cloudflared tunnel --url"; URL đọc từ file log
_tlog = open("/tmp/cloudflared.log", "w")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                 stdout=_tlog, stderr=subprocess.STDOUT, text=True, start_new_session=True)
url = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/tmp/cloudflared.log").read())
    if m: url = m.group(0); break
print("🌐 UI demo:", url or "KHÔNG LẤY ĐƯỢC URL — xem /tmp/cloudflared.log")
if url:
    open("/tmp/demo_url.txt", "w").write(url)

## 7. Smoke-test trên T4

In [ ]:
%cd /content/Reproduce_Paths-to-Causality_KDD-25/demo
import sys, os, time
sys.path.insert(0, ".")
os.environ.setdefault("NEO4J_URI", "bolt://localhost:7687")

import torch
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

from src.repo_ce_ranker import discover_best_ce, RepoCrossEncoderRanker
best = discover_best_ce("models/ce")
print("Checkpoint tốt nhất:", best)
ce = RepoCrossEncoderRanker.try_load(best)
assert ce is not None, "CE không load được — xem thông báo ở trên (thiếu torch/transformers?)"
print(f"Load xong | device={ce.device}")
paths = [
    {"path_nodes": [{"name": "TRPV6", "kind": "Gene"}, {"name": "TRAF1", "kind": "Gene"}, {"name": "prostate cancer", "kind": "Disease"}]},
    {"path_nodes": [{"name": "FGF6", "kind": "Gene"}, {"name": "prostate cancer", "kind": "Disease"}]},
]
t0 = time.time(); n = len(paths) * 8
ce.score("FGF6", "prostate cancer", paths * 8)
dt = time.time() - t0
print(f"CE inference: {n} paths / {dt:.2f}s ({dt/n*1000:.0f} ms/path)")

from src.ranker import PathRanker
print("PathRanker mode:", PathRanker().mode)
for v, lbl in PathRanker.choices():
    print("  option:", v, "|", lbl[:70])

In [ ]:
# End-to-end: 1 sample COMAGC thật qua Hetionet -> CE -> Ollama
import json
row = json.loads(open("data/benchmark/comagc/processed_test_truth.jsonl", encoding="utf-8").readline())
sample = dict(row); sample["dataset"] = "comagc"; sample["source_file"] = "test_truth.jsonl"
e1 = {"id": "x", "name": row["e1"], "kind": "Gene"}
e2 = {"id": "y", "name": row["e2"], "kind": "Disease"}
print("Cặp:", row["e1"], "→", row["e2"])

from src.pipeline import analyze_pair
try:
    r = analyze_pair(e1, e2, row["sentence"][:200], mode="paper", sample=sample)
    print("path_source:", r["path_source"], "| paths:", r["paths_found"])
    print("ranker_mode:", r["ranker_mode"])
    print("prediction:", r["prediction"]["label"])
except Exception as e:
    print(f"Lỗi ({type(e).__name__}): {e}")
    print("-> Kiểm tra: Neo4j sống chưa (cell 4)? Ollama sống chưa (chạy lại cell 5)?")

## Ghi chú
- **URL tunnel** chỉ sống trong phiên Colab; nếu mất, chạy lại cell 6 để lấy URL mới (các bước trước không cần làm lại).
- UI: sidebar **Ranking model** liệt kê các checkpoint cross-encoder kèm NDCG@5 (chạy trên T4); **Pipeline LLM** đổi sang OpenAI-compatible (OpenRouter…) nếu muốn LLM khác qwen2.5:7b.
- Process nền (Neo4j/Ollama/Streamlit/cloudflared) đều start detached — không bị giết khi cell kết thúc, và mỗi cell chạy lại được an toàn.
- Ground truth không gửi vào prompt; nhãn causal/non-causal chỉ là dự đoán của model.